# UM Building Footprints GeoJSON Cleaner

### This notebook:
- Loads the UM building footprints GeoJSON
- Simplifies geometries for web mapping performance
- Drops unneeded GIS-only metadata fields
- Creates and normalizes AI resource column into a boolean
- Outputs a cleaned geoJSON

In [1]:
import pandas as pd
import geopandas as gpd

In [2]:
# Load and inspect the data
building_footprint = gpd.read_file('um-building-footprint.geojson')
print("rows:", len(building_footprint))
print("crs:", building_footprint.crs) #Checks for coordinate reference system
print("geometry types:\n", building_footprint.geom_type.value_counts()) #Some multipolygons, will need to be converted to polygons
building_footprint.head()

rows: 1042
crs: EPSG:4326
geometry types:
 Polygon         1007
MultiPolygon      35
Name: count, dtype: int64


,OBJECTID,loc_Campus_descr,loc_SubCampus_descr,loc_MainCampus,Status,Ownership,PanelType,ObjectName,Building_Type,loc_ObjectNum,created_user,created_date,last_edited_user,last_edited_date,GlobalID,SUBTYPE,Shape__Area,Shape__Length,geometry
0,1,Ann Arbor,North Campus,Main,Active,Own,Building,NCRC B500 (Underground),BLDGU,1005280,brimcel_umich,1751976683021,brimcel_umich,1751976683021,5022191f-82a9-4b0d-9ae3-8fe1fac098cf,3,15083.046875,1214.576194,"POLYGON ((-83.70362 42.2999, -83.70362 42.2999..."
1,2,Ann Arbor,North Campus,Main,Active,Own,Building,NCRC Bldg 15,BLDGU,1005255,brimcel_umich,1751976683021,brimcel_umich,1751976683021,ff531986-1ca8-40a8-aff9-a3ae7b7670d9,3,5937.982910,322.868746,"POLYGON ((-83.7067 42.30023, -83.7067 42.30004..."
2,3,Ann Arbor,North Campus,Main,Active,Own,Building,NCRC Bldg 23,BLDGU,1005261,brimcel_umich,1751976683021,brimcel_umich,1751976683021,492056f7-f411-46da-8094-f323a85481a3,3,10717.561279,643.277263,"POLYGON ((-83.70674 42.29968, -83.70674 42.299..."
3,4,Ann Arbor,Central Campus,Main,Active,Own,Building,Central Campus Support Facility,BLDGU,1005379,brimcel_umich,1751976683021,brimcel_umich,1751976683021,3a10f37e-15f9-41cf-bfb6-8027c21d1bad,3,87.915283,39.349791,"POLYGON ((-83.7354 42.27905, -83.7354 42.27907..."
4,6,Ann Arbor,Medical Campus,Main,Active,Own,Building,Cardiovascular Center Parking Structure,PARKU,1005146,brimcel_umich,1751976683021,brimcel_umich,1751976683021,a314aa07-9fc4-40aa-9e73-18c9827a08e2,3,75928.744141,1362.263516,"POLYGON ((-83.73085 42.28281, -83.73082 42.282..."


In [3]:
# Dropping unnecessary columns
keep_cols = [
    "ObjectName",
    "loc_ObjectNum",
    "loc_Campus_descr",
    "loc_SubCampus_descr",
    "GlobalID"
]

gdf = building_footprint[keep_cols + ["geometry"]].copy()
#gdf.head(2)

In [4]:
# Rename columns to be cleaner and consistent/contextual
names = {
    "ObjectName": "building_name",
    "loc_ObjectNum": "building_id",
    "loc_Campus_descr": "campus",
    "loc_SubCampus_descr": "sub_campus",
}

gdf = gdf.rename(columns={k:v for k,v in names.items() if k in gdf.columns})
#gdf.head(2)

In [5]:
resources = {
    "Vaughan SPH-I", #SABER
    "Institute for Social Research", #d3 ---- #ESC
    "NCRC Bldg 550", #DISC
    "NCRC Bldg 520", #e-HAIL
    "NCRC Bldg 100", #AI & Digital Health Innovation ---- #Center for Global Health Equity
    "Beyster Building", #AI Lab
    "3520 Green Court", #Institute for Comp. Discovery and Engineering
    "Alexander G. Ruthven Building", #ARIA ---- #Bold Challenges
    "East Hall", #Center for Applied and Interdisc. Mathematics
    "Mason Hall", #Digital Studies Institute
    "Weill Hall" #STPP Program
    "Weiser Hall" #MIDAS

}

# Still to find a footprint for:
# ARC
# Alzeimer's Disease Research Center
# AIIM

In [ ]:
# Output finalized geoJSON
building_footprint.to_file("um-building-footprint-edited.geojson", driver="GeoJSON")